In [1]:
pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


ERROR: After October 2020 you may experience errors when installing or updating packages. This is because pip will change the way that it resolves dependency conflicts.

We recommend you use --use-feature=2020-resolver to test your packages with the new resolver before it becomes the default.

selenium 4.26.1 requires urllib3[socks]<3,>=1.26, but you'll have urllib3 1.25.11 which is incompatible.


  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3


In [3]:
import os
import re
import time
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- 設定 ---
BASE_URL = "https://shiny-ace.com/"
TXT_FILE = "リンク一覧.txt"
CSV_FILE = "fish_list.csv"
IMAGE_DIR = "downloaded_images"

# 画像保存用のフォルダを作成
os.makedirs(IMAGE_DIR, exist_ok=True)

# リンク一覧から「zukan/分類/ページ名.html」のパターンを抽出する正規表現
url_pattern = re.compile(r'(zukan\d*/[^/]+/[^/]+\.html)')

# --- 1. リンク一覧の読み込みとパース ---
print("リンク一覧を読み込んでいます...")
extracted_paths = []

with open(TXT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        match = url_pattern.search(line)
        if match:
            extracted_paths.append(match.group(1))

print(f"合計 {len(extracted_paths)} 件のページを検出しました。")

# --- 2. スクレイピングとダウンロードの実行 ---
csv_headers = ["魚の名前", "英名", "保存した画像ファイル名", "分類", "解説"]

# Excelの文字化け対策で 'utf-8-sig' を指定
with open(CSV_FILE, 'w', encoding='utf-8-sig', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(csv_headers)

    for i, path in enumerate(extracted_paths, 1):
        parts = path.split('/')
        category = parts[-2]  # URLの1つ上の階層（分類）

        page_url = BASE_URL + path
        print(f"[{i}/{len(extracted_paths)}] 処理中: {page_url}")

        # 初期値（情報が取得できない場合は空文字または欠損とする）
        fish_name = ""
        english_name = ""
        saved_image_name = "欠損"
        description = ""

        try:
            # 魚の個別ページを取得
            res = requests.get(page_url, timeout=10)
            res.raise_for_status()
            res.encoding = 'utf-8'
            
            soup = BeautifulSoup(res.text, 'html.parser')

            # 魚の名前・英名の取得
            h2_tag = soup.find('h2')
            if h2_tag:
                h2_text = h2_tag.get_text()
                if "英名：" in h2_text:
                    h2_parts = h2_text.split("英名：")
                    fish_name = h2_parts[0].strip().replace("　", "")
                    english_name = h2_parts[1].strip()
                else:
                    fish_name = h2_text.strip()
            
            if not fish_name:
                h1_tag = soup.find('h1')
                if h1_tag:
                    fish_name = h1_tag.contents[0].strip() if h1_tag.contents else h1_tag.get_text().strip()

            # 解説文の取得
            desc_div = soup.find('div', class_='notedesign1')
            if desc_div:
                description = desc_div.get_text(separator=' ', strip=True)

            # --- [変更点] figure.gallery内の最初の画像を取得 ---
            figure_tag = soup.find('figure', class_='gallery')
            if figure_tag:
                img_tag = figure_tag.find('img')
                if img_tag and img_tag.get('src'):
                    img_src = img_tag.get('src')
                    
                    # サイト上の相対パス（または絶対パス）を絶対URLに変換
                    image_url = urljoin(page_url, img_src)
                    # URLの末尾から実際の画像ファイル名を抽出（例: akaei4.jpg）
                    image_name = image_url.split('/')[-1]

                    # 画像のダウンロードを実行
                    img_res = requests.get(image_url, timeout=10)
                    if img_res.status_code == 200:
                        img_path = os.path.join(IMAGE_DIR, image_name)
                        with open(img_path, 'wb') as img_f:
                            img_f.write(img_res.content)
                        saved_image_name = image_name
                    else:
                        print(f"  --> [画像取得失敗] ステータス: {img_res.status_code} ({image_url})")
                else:
                    print("  --> [画像タグなし] figure内にimgタグが見つかりません。")
            else:
                print("  --> [要素なし] figure.gallery が見つかりません。")

        except Exception as e:
            print(f"  --> [エラー発生] {page_url} の処理中に問題が発生しました: {e}")

        # CSVに1行書き込み
        writer.writerow([fish_name, english_name, saved_image_name, category, description])
        
        # インターバルを0.1秒に変更
        time.sleep(0.1)

print("\nすべての処理が完了しました！")
print(f"CSVファイル: {CSV_FILE}")
print(f"画像格納フォルダ: {IMAGE_DIR}/")

リンク一覧を読み込んでいます...
合計 1323 件のページを検出しました。
[1/1323] 処理中: https://shiny-ace.com/zukan/ei/akaei.html
[2/1323] 処理中: https://shiny-ace.com/zukan/ei/yakkoei.html
[3/1323] 処理中: https://shiny-ace.com/zukan/beraka/ira.html
[4/1323] 処理中: https://shiny-ace.com/zukan/ajika/buri.html
[5/1323] 処理中: https://shiny-ace.com/zukan/hataka/kue.html
[6/1323] 処理中: https://shiny-ace.com/zukan/boraka/bora.html
[7/1323] 処理中: https://shiny-ace.com/zukan/ajika/maaji.html
[8/1323] 処理中: https://shiny-ace.com/zukan/taika/madai.html
[9/1323] 処理中: https://shiny-ace.com/zukan/ei/madaraei.html
[10/1323] 処理中: https://shiny-ace.com/zukan/ei/hirataei.html
[11/1323] 処理中: https://shiny-ace.com/zukan/ei/mobula1-.html
[12/1323] 処理中: https://shiny-ace.com/zukan/ajika/meaji.html
[13/1323] 処理中: https://shiny-ace.com/zukan/sugika/sugi.html
[14/1323] 処理中: https://shiny-ace.com/zukan/sabaka/cero.html
[15/1323] 処理中: https://shiny-ace.com/zukan/aigoka/aigo.html
[16/1323] 処理中: https://shiny-ace.com/zukan/esoka/akaeso.html
[17/1323] 処理中: 

[128/1323] 処理中: https://shiny-ace.com/zukan/beraka/otomebera.html
[129/1323] 処理中: https://shiny-ace.com/zukan/esoka/hoshinoeso.html
[130/1323] 処理中: https://shiny-ace.com/zukan/hazeka/oomonhaze.html
[131/1323] 処理中: https://shiny-ace.com/zukan/beraka/ogurobera.html
[132/1323] 処理中: https://shiny-ace.com/zukan/hazeka/kobanhaze.html
[133/1323] 処理中: https://shiny-ace.com/zukan/gonbeka/okigonbe.html
[134/1323] 処理中: https://shiny-ace.com/zukan/hataka/oomonhata.html
[135/1323] 処理中: https://shiny-ace.com/zukan/hazeka/colongoby.html
[136/1323] 処理中: https://shiny-ace.com/zukan/hazeka/oiranhaze.html
[137/1323] 処理中: https://shiny-ace.com/zukan/hataka/sakuradai.html
[138/1323] 処理中: https://shiny-ace.com/zukan/kisuka/shirogisu.html
[139/1323] 処理中: https://shiny-ace.com/zukan/beraka/yashabera.html
[140/1323] 処理中: https://shiny-ace.com/zukan/hazeka/odorihaze.html
[141/1323] 処理中: https://shiny-ace.com/zukan/hataka/houkihata.html
[142/1323] 処理中: https://shiny-ace.com/zukan/ajika/onihiraaji.html
[143/1323]

[252/1323] 処理中: https://shiny-ace.com/zukan/beraka/kujakubera.html
[253/1323] 処理中: https://shiny-ace.com/zukan/beraka/kusabibera.html
[254/1323] 処理中: https://shiny-ace.com/zukan/kochika/wanigochi.html
[255/1323] 処理中: https://shiny-ace.com/zukan/hazeka/noborihaze.html
[256/1323] 処理中: https://shiny-ace.com/zukan/himejika/umihigoi.html
[257/1323] 処理中: https://shiny-ace.com/zukan/ajika/hosohiraaji.html
  --> [要素なし] figure.gallery が見つかりません。
[258/1323] 処理中: https://shiny-ace.com/zukan/hataka/yukatahata.html
[259/1323] 処理中: https://shiny-ace.com/zukan/hataka/madarahata.html
[260/1323] 処理中: https://shiny-ace.com/zukan/budaika/sujibudai.html
[261/1323] 処理中: https://shiny-ace.com/zukan/beraka/manabebera.html
[262/1323] 処理中: https://shiny-ace.com/zukan/hazeka/madarahaze.html
[263/1323] 処理中: https://shiny-ace.com/zukan/hazeka/nagasehaze.html
[264/1323] 処理中: https://shiny-ace.com/zukan/hataka/sarasahata.html
[265/1323] 処理中: https://shiny-ace.com/zukan/hataka/aonomehata.html
[266/1323] 処理中: https://

[371/1323] 処理中: https://shiny-ace.com/zukan/ajika/nanyoukaiwari.html
[372/1323] 処理中: https://shiny-ace.com/zukan/budaika/nanyoubudai.html
[373/1323] 処理中: https://shiny-ace.com/zukan/utsuboka/kokeutsubo.html
[374/1323] 処理中: https://shiny-ace.com/zukan/hazeka/shimaorihaze.html
[375/1323] 処理中: https://shiny-ace.com/zukan/hataka/tsuchihozeri.html
[376/1323] 処理中: https://shiny-ace.com/zukan/hazeka/goldspotgoby.html
[377/1323] 処理中: https://shiny-ace.com/zukan/ittodaika/nijiebisu.html
[378/1323] 処理中: https://shiny-ace.com/zukan/hazeka/hagoromohaze.html
[379/1323] 処理中: https://shiny-ace.com/zukan/nizadaika/tenguhagi.html
[380/1323] 処理中: https://shiny-ace.com/zukan/hakofuguka/hakofugu.html
[381/1323] 処理中: https://shiny-ace.com/zukan/beraka/hagehirabera.html
[382/1323] 処理中: https://shiny-ace.com/zukan2/kamemoku/aoumigame.html
[383/1323] 処理中: https://shiny-ace.com/zukan/hazeka/garasuhaze5-.html
[384/1323] 処理中: https://shiny-ace.com/zukan/himejika/indohimeji.html
[385/1323] 処理中: https://shiny-ace.

[488/1323] 処理中: https://shiny-ace.com/zukan/hazeka/ichimonjihaze.html
[489/1323] 処理中: https://shiny-ace.com/zukan/fuedaika/ittenfuedai.html
[490/1323] 処理中: https://shiny-ace.com/zukan/nizadaika/nanyouhagi.html
[491/1323] 処理中: https://shiny-ace.com/zukan/hataka/niramihanadai.html
[492/1323] 処理中: https://shiny-ace.com/zukan/budaika/kanmuribudai.html
[493/1323] 処理中: https://shiny-ace.com/zukan/umitanagoka/aotanago.html
[494/1323] 処理中: https://shiny-ace.com/zukan/hataka/panamagraysby.html
[495/1323] 処理中: https://shiny-ace.com/zukan/same/oguromejirozame.html
[496/1323] 処理中: https://shiny-ace.com/zukan/hataka/osyarehanadai.html
[497/1323] 処理中: https://shiny-ace.com/zukan/hazeka/blackeyedgoby.html
[498/1323] 処理中: https://shiny-ace.com/zukan/beraka/kazarikyuusen.html
[499/1323] 処理中: https://shiny-ace.com/zukan/hataka/harlequinbass.html
[500/1323] 処理中: https://shiny-ace.com/zukan/beraka/kisujikyuusen.html
[501/1323] 処理中: https://shiny-ace.com/zukan/hazeka/obakeinkohaze.html
[502/1323] 処理中: http

...

[828/1323] 処理中: https://shiny-ace.com/zukan/hazeka/ichimonjikobanhaze.html
[829/1323] 処理中: https://shiny-ace.com/zukan/kagokakidaika/kagokakidai.html
[830/1323] 処理中: https://shiny-ace.com/zukan/utsuboka/herigoishiutsubo.html
[831/1323] 処理中: https://shiny-ace.com/zukan/hazeka/kataboshioomonhaze.html
[832/1323] 処理中: https://shiny-ace.com/zukan/takasagoka/bananafusilier.html
[833/1323] 処理中: https://shiny-ace.com/zukan/fuguka/kazarikinchakufugu.html
[834/1323] 処理中: https://shiny-ace.com/zukan/harisenbonka/ishigakifugu.html
[835/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/rurisuzumedai.html
[836/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/oogonnijiginpo.html
[837/1323] 処理中: https://shiny-ace.com/zukan/beraka/munatenberadamashi.html
[838/1323] 処理中: https://shiny-ace.com/zukan/toragisuka/madaratoragisu.html
[839/1323] 処理中: https://shiny-ace.com/zukan/hazeka/candycanedwarfgoby.html
[840/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/ukkarikasago.html
[841/1323] 処理中: https://s

[937/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/asadosuzumedai.html
[938/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/nishikiyakko.html
[939/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/kakurekumanomi.html
[940/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/touakakumanomi.html
[941/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/akaharayakko.html
[942/1323] 処理中: https://shiny-ace.com/zukan/mongarakawahagika/kumadori.html
[943/1323] 処理中: https://shiny-ace.com/zukan/ittodaika/kuroobomatsukasa.html
[944/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/segmentedblenny.html
[945/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/inadumayakko.html
[946/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/ishigakikaeruuo.html
[947/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/tategamikaeruuo.html
[948/1323] 処理中: https://shiny-ace.com/zukan/utsuboka/panamicgreenmoray.html
[949/1323] 処理中: https://shiny-ace.com/zukan/kintokidaika/minamikintoki.html
[950/1323] 処

[1044/1323] 処理中: https://shiny-ace.com/zukan/kokeginpoka/papilloseblenny.html
[1045/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/blackspotdamsel.html
[1046/1323] 処理中: https://shiny-ace.com/zukan/kuroyurihazeka/kuroyurihaze.html
[1047/1323] 処理中: https://shiny-ace.com/zukan/manjuudaika/nanyoutsubameuo.html
[1048/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/goldbellydamsel.html
[1049/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/coraldemoiselle.html
[1050/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/koganesuzumedai.html
[1051/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/hanaminokasago.html
[1052/1323] 処理中: https://shiny-ace.com/zukan/agoamadaika/variablejawfish.html
[1053/1323] 処理中: https://shiny-ace.com/zukan/kokeginpoka/hadakakokeginpo.html
[1054/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/zebraangelfish.html
[1055/1323] 処理中: https://shiny-ace.com/zukan/mongarakawahagika/akamongara.html
[1056/1323] 処理中: https://shiny-ace.com/zukan/beraka/sumitsukis

[1148/1323] 処理中: https://shiny-ace.com/zukan/chouchouuoka/kagamichouchouuo.html
[1149/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/frenchangelfish.html
[1150/1323] 処理中: https://shiny-ace.com/zukan/fuefukidaika/yokoshimakurodai.html
[1151/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/nettaiminokasago.html
[1152/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sangiruishimochi.html
[1153/1323] 処理中: https://shiny-ace.com/zukan/chouchouuoka/sudarechouchouuo.html
[1154/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/cortezangelfish.html
[1155/1323] 処理中: https://shiny-ace.com/zukan/chouchouuoka/fuuraichouchouuo.html
[1156/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/talbotsdemoiselle.html
[1157/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/hirenagasuzumedai.html
[1158/1323] 処理中: https://shiny-ace.com/zukan/nizadaika/gomatenguhagimodoki.html
[1159/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/hiregurosuzumedai.html
[1160/1323] 処理中: https://shiny-ace.com/z

[1249/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/miyakokisensuzumedai.html
[1250/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/yellowsidedamselfish.html
[1251/1323] 処理中: https://shiny-ace.com/zukan2/kujiraguuteimoku/hashinagairuka.html
[1252/1323] 処理中: https://shiny-ace.com/zukan/kochika/indianoceancrocodilefish.html
[1253/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/shimahimeyamanokami.html
[1254/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/allenscarodinalfish.html
[1255/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai3-.html
[1256/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai1-.html
[1257/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/shirikirurisuzumedai.html
[1258/1323] 処理中: https://shiny-ace.com/zukan/kuroyurihazeka/ogurokuroyurihaze.html
[1259/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai2-.html
[1260/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/tatejimakinchakudai.html
[12

In [5]:
import os
import re
import time
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- 設定 ---
BASE_URL = "https://shiny-ace.com/"
TXT_FILE = "リンク一覧.txt"
CSV_FILE = "fish_list.csv"  # 日本語書き換え・並び替え済みのCSV

IMAGE_DIR2 = "downloaded_images2"
IMAGE_DIR3 = "downloaded_images3"

# 新しい画像保存用のフォルダを作成
os.makedirs(IMAGE_DIR2, exist_ok=True)
os.makedirs(IMAGE_DIR3, exist_ok=True)

# リンク一覧から「zukan/分類/ページ名.html」のパターンを抽出する正規表現
url_pattern = re.compile(r'(zukan\d*/[^/]+/[^/]+\.html)')

# --- 1. リンク一覧の読み込み ---
print("リンク一覧を読み込んでいます...")
extracted_paths = []
with open(TXT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        match = url_pattern.search(line)
        if match:
            extracted_paths.append(match.group(1))
print(f"合計 {len(extracted_paths)} 件のページを検出しました。")


# --- 2. スクレイピングの実行（魚の名前をキーにして画像名を記憶） ---
scraped_images = {}

print("\n2枚目・3枚目の画像スクレイピングを開始します...")
for i, path in enumerate(extracted_paths, 1):
    page_url = BASE_URL + path
    print(f"[{i}/{len(extracted_paths)}] 処理中: {page_url}")

    try:
        res = requests.get(page_url, timeout=10)
        res.raise_for_status()
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')

        # 魚の名前の取得（既存の1枚目のロジックと完全に統一）
        fish_name = ""
        h2_tag = soup.find('h2')
        if h2_tag:
            h2_text = h2_tag.get_text()
            if "英名：" in h2_text:
                h2_parts = h2_text.split("英名：")
                fish_name = h2_parts[0].strip().replace("　", "")
            else:
                fish_name = h2_text.strip()
        
        if not fish_name:
            h1_tag = soup.find('h1')
            if h1_tag:
                fish_name = h1_tag.contents[0].strip() if h1_tag.contents else h1_tag.get_text().strip()

        # 名前が取得できなかった場合はログを出してスキップ
        if not fish_name:
            print(f"  --> [警告] 魚の名前が取得できませんでした: {page_url}")
            continue

        image2_name = "欠損"
        image3_name = "欠損"

        # すべてのfigure.galleryタグを取得
        figure_tags = soup.find_all('figure', class_='gallery')
        gallery_imgs = []
        for fig in figure_tags:
            img_tag = fig.find('img')
            if img_tag and img_tag.get('src'):
                gallery_imgs.append(img_tag.get('src'))

        # 2番目の画像のダウンロード
        if len(gallery_imgs) > 1:
            img_src2 = gallery_imgs[1]
            image_url2 = urljoin(page_url, img_src2)
            image2_name = image_url2.split('/')[-1]

            img_res2 = requests.get(image_url2, timeout=10)
            if img_res2.status_code == 200:
                with open(os.path.join(IMAGE_DIR2, image2_name), 'wb') as img_f:
                    img_f.write(img_res2.content)
            else:
                image2_name = "欠損"

        # 3番目の画像のダウンロード
        if len(gallery_imgs) > 2:
            img_src3 = gallery_imgs[2]
            image_url3 = urljoin(page_url, img_src3)
            image3_name = image_url3.split('/')[-1]

            img_res3 = requests.get(image_url3, timeout=10)
            if img_res3.status_code == 200:
                with open(os.path.join(IMAGE_DIR3, image3_name), 'wb') as img_f:
                    img_f.write(img_res3.content)
            else:
                image3_name = "欠損"

        # 【ここがポイント】抽出した正確な「魚の名前」をキーにして画像名を辞書に保存
        scraped_images[fish_name] = (image2_name, image3_name)

    except Exception as e:
        print(f"  --> [エラー発生] {page_url} の処理中に問題が発生しました: {e}")

    time.sleep(0.1)


# --- 3. 既存の並び替え済みCSVを読み込み、名前が一致する行にマッピングして上書き ---
print("\n既存のCSVファイルに画像データをマッピングしています...")

updated_rows = []
new_header = ["魚の名前", "英名", "保存した画像ファイル名", "分類", "解説", "image2", "image3"]

try:
    with open(CSV_FILE, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        existing_header = next(reader)  # 既存のヘッダーをスキップ
        
        for row in reader:
            if not row:
                continue
            
            # 既存CSVの1列目から魚の名前を取得（念のためスペースを除去）
            current_fish_name = row[0].strip().replace("　", "")
            
            # スクレイピング結果の辞書から、名前が完全に一致する画像名を取り出す
            if current_fish_name in scraped_images:
                img2, img3 = scraped_images[current_fish_name]
            else:
                img2, img3 = "欠損", "欠損"
            
            # 前回の失敗で古いimage2,3列がすでに右端にくっついてしまっていても、
            # row[:5] で最初の5列（名前、英名、画像名、分類、解説）だけを綺麗に切り出して、新しい正しい画像名を結合します
            base_data = row[:5]
            while len(base_data) < 5:
                base_data.append("")  # 万が一、データが欠けている行があった場合の安全対策
                
            updated_rows.append(base_data + [img2, img3])

    # 元の並び順のまま安全に上書き保存
    with open(CSV_FILE, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(new_header)
        writer.writerows(updated_rows)

    print("\n[完了] 魚の名前との紐付けが完了し、CSVが安全に更新されました！")

except FileNotFoundError:
    print(f"エラー: {CSV_FILE} が見つかりません。既存のCSVがあるフォルダで実行してください。")

リンク一覧を読み込んでいます...
合計 1323 件のページを検出しました。
既存のCSVファイルを読み込んで保護しています...

2枚目・3枚目の画像スクレイピングを開始します...
[1/1323] 処理中: https://shiny-ace.com/zukan/ei/akaei.html
[2/1323] 処理中: https://shiny-ace.com/zukan/ei/yakkoei.html
[3/1323] 処理中: https://shiny-ace.com/zukan/beraka/ira.html
[4/1323] 処理中: https://shiny-ace.com/zukan/ajika/buri.html
[5/1323] 処理中: https://shiny-ace.com/zukan/hataka/kue.html
[6/1323] 処理中: https://shiny-ace.com/zukan/boraka/bora.html
[7/1323] 処理中: https://shiny-ace.com/zukan/ajika/maaji.html
[8/1323] 処理中: https://shiny-ace.com/zukan/taika/madai.html
[9/1323] 処理中: https://shiny-ace.com/zukan/ei/madaraei.html
[10/1323] 処理中: https://shiny-ace.com/zukan/ei/hirataei.html
[11/1323] 処理中: https://shiny-ace.com/zukan/ei/mobula1-.html
[12/1323] 処理中: https://shiny-ace.com/zukan/ajika/meaji.html
[13/1323] 処理中: https://shiny-ace.com/zukan/sugika/sugi.html
[14/1323] 処理中: https://shiny-ace.com/zukan/sabaka/cero.html
[15/1323] 処理中: https://shiny-ace.com/zukan/aigoka/aigo.html
[16/1323] 処理中: https:

[127/1323] 処理中: https://shiny-ace.com/zukan/hazeka/otomehaze.html
[128/1323] 処理中: https://shiny-ace.com/zukan/beraka/otomebera.html
[129/1323] 処理中: https://shiny-ace.com/zukan/esoka/hoshinoeso.html
[130/1323] 処理中: https://shiny-ace.com/zukan/hazeka/oomonhaze.html
[131/1323] 処理中: https://shiny-ace.com/zukan/beraka/ogurobera.html
[132/1323] 処理中: https://shiny-ace.com/zukan/hazeka/kobanhaze.html
[133/1323] 処理中: https://shiny-ace.com/zukan/gonbeka/okigonbe.html
[134/1323] 処理中: https://shiny-ace.com/zukan/hataka/oomonhata.html
[135/1323] 処理中: https://shiny-ace.com/zukan/hazeka/colongoby.html
[136/1323] 処理中: https://shiny-ace.com/zukan/hazeka/oiranhaze.html
[137/1323] 処理中: https://shiny-ace.com/zukan/hataka/sakuradai.html
[138/1323] 処理中: https://shiny-ace.com/zukan/kisuka/shirogisu.html
[139/1323] 処理中: https://shiny-ace.com/zukan/beraka/yashabera.html
[140/1323] 処理中: https://shiny-ace.com/zukan/hazeka/odorihaze.html
[141/1323] 処理中: https://shiny-ace.com/zukan/hataka/houkihata.html
[142/1323]

[251/1323] 処理中: https://shiny-ace.com/zukan/gonbeka/kudagonbe.html
[252/1323] 処理中: https://shiny-ace.com/zukan/beraka/kujakubera.html
[253/1323] 処理中: https://shiny-ace.com/zukan/beraka/kusabibera.html
[254/1323] 処理中: https://shiny-ace.com/zukan/kochika/wanigochi.html
[255/1323] 処理中: https://shiny-ace.com/zukan/hazeka/noborihaze.html
[256/1323] 処理中: https://shiny-ace.com/zukan/himejika/umihigoi.html
[257/1323] 処理中: https://shiny-ace.com/zukan/ajika/hosohiraaji.html
[258/1323] 処理中: https://shiny-ace.com/zukan/hataka/yukatahata.html
[259/1323] 処理中: https://shiny-ace.com/zukan/hataka/madarahata.html
[260/1323] 処理中: https://shiny-ace.com/zukan/budaika/sujibudai.html
[261/1323] 処理中: https://shiny-ace.com/zukan/beraka/manabebera.html
[262/1323] 処理中: https://shiny-ace.com/zukan/hazeka/madarahaze.html
[263/1323] 処理中: https://shiny-ace.com/zukan/hazeka/nagasehaze.html
[264/1323] 処理中: https://shiny-ace.com/zukan/hataka/sarasahata.html
[265/1323] 処理中: https://shiny-ace.com/zukan/hataka/aonomehata.

[372/1323] 処理中: https://shiny-ace.com/zukan/budaika/nanyoubudai.html
[373/1323] 処理中: https://shiny-ace.com/zukan/utsuboka/kokeutsubo.html
[374/1323] 処理中: https://shiny-ace.com/zukan/hazeka/shimaorihaze.html
[375/1323] 処理中: https://shiny-ace.com/zukan/hataka/tsuchihozeri.html
[376/1323] 処理中: https://shiny-ace.com/zukan/hazeka/goldspotgoby.html
[377/1323] 処理中: https://shiny-ace.com/zukan/ittodaika/nijiebisu.html
[378/1323] 処理中: https://shiny-ace.com/zukan/hazeka/hagoromohaze.html
[379/1323] 処理中: https://shiny-ace.com/zukan/nizadaika/tenguhagi.html
[380/1323] 処理中: https://shiny-ace.com/zukan/hakofuguka/hakofugu.html
[381/1323] 処理中: https://shiny-ace.com/zukan/beraka/hagehirabera.html
[382/1323] 処理中: https://shiny-ace.com/zukan2/kamemoku/aoumigame.html
[383/1323] 処理中: https://shiny-ace.com/zukan/hazeka/garasuhaze5-.html
[384/1323] 処理中: https://shiny-ace.com/zukan/himejika/indohimeji.html
[385/1323] 処理中: https://shiny-ace.com/zukan/hazeka/yamabukihaze.html
[386/1323] 処理中: https://shiny-ace.

[491/1323] 処理中: https://shiny-ace.com/zukan/hataka/niramihanadai.html
[492/1323] 処理中: https://shiny-ace.com/zukan/budaika/kanmuribudai.html
[493/1323] 処理中: https://shiny-ace.com/zukan/umitanagoka/aotanago.html
[494/1323] 処理中: https://shiny-ace.com/zukan/hataka/panamagraysby.html
[495/1323] 処理中: https://shiny-ace.com/zukan/same/oguromejirozame.html
[496/1323] 処理中: https://shiny-ace.com/zukan/hataka/osyarehanadai.html
[497/1323] 処理中: https://shiny-ace.com/zukan/hazeka/blackeyedgoby.html
[498/1323] 処理中: https://shiny-ace.com/zukan/beraka/kazarikyuusen.html
[499/1323] 処理中: https://shiny-ace.com/zukan/hataka/harlequinbass.html
[500/1323] 処理中: https://shiny-ace.com/zukan/beraka/kisujikyuusen.html
[501/1323] 処理中: https://shiny-ace.com/zukan/hazeka/obakeinkohaze.html
[502/1323] 処理中: https://shiny-ace.com/zukan/isakika/blackmargate.html
[503/1323] 処理中: https://shiny-ace.com/zukan/beraka/hokurokyuusen.html
[504/1323] 処理中: https://shiny-ace.com/zukan/hazeka/kuroitohaze1-.html
[505/1323] 処理中: http

...

[833/1323] 処理中: https://shiny-ace.com/zukan/fuguka/kazarikinchakufugu.html
[834/1323] 処理中: https://shiny-ace.com/zukan/harisenbonka/ishigakifugu.html
[835/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/rurisuzumedai.html
[836/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/oogonnijiginpo.html
[837/1323] 処理中: https://shiny-ace.com/zukan/beraka/munatenberadamashi.html
[838/1323] 処理中: https://shiny-ace.com/zukan/toragisuka/madaratoragisu.html
[839/1323] 処理中: https://shiny-ace.com/zukan/hazeka/candycanedwarfgoby.html
[840/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/ukkarikasago.html
[841/1323] 処理中: https://shiny-ace.com/zukan/matsukasauoka/matsukasauo.html
[842/1323] 処理中: https://shiny-ace.com/zukan/isakika/hiregurokosyoudai.html
[843/1323] 処理中: https://shiny-ace.com/zukan/ittodaika/hireguroittodai.html
[844/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/kinchakudai.html
[845/1323] 処理中: https://shiny-ace.com/zukan/ittodaika/ukeguchiittodai.html
[846/1323] 処理中: https://s

[942/1323] 処理中: https://shiny-ace.com/zukan/mongarakawahagika/kumadori.html
[943/1323] 処理中: https://shiny-ace.com/zukan/ittodaika/kuroobomatsukasa.html
[944/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/segmentedblenny.html
[945/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/inadumayakko.html
[946/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/ishigakikaeruuo.html
[947/1323] 処理中: https://shiny-ace.com/zukan/isoginpoka/tategamikaeruuo.html
[948/1323] 処理中: https://shiny-ace.com/zukan/utsuboka/panamicgreenmoray.html
[949/1323] 処理中: https://shiny-ace.com/zukan/kintokidaika/minamikintoki.html
[950/1323] 処理中: https://shiny-ace.com/zukan/kaeruankouka/irokaeruankou.html
[951/1323] 処理中: https://shiny-ace.com/zukan/nezuppoka/minamikobunumeri.html
[952/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/imitatordamsel.html
[953/1323] 処理中: https://shiny-ace.com/zukan/asahiginpoka/giantkelpfish.html
[954/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/alphasuzumedai.html
[955/1323] 処

[1049/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/coraldemoiselle.html
[1050/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/koganesuzumedai.html
[1051/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/hanaminokasago.html
[1052/1323] 処理中: https://shiny-ace.com/zukan/agoamadaika/variablejawfish.html
[1053/1323] 処理中: https://shiny-ace.com/zukan/kokeginpoka/hadakakokeginpo.html
[1054/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/zebraangelfish.html
[1055/1323] 処理中: https://shiny-ace.com/zukan/mongarakawahagika/akamongara.html
[1056/1323] 処理中: https://shiny-ace.com/zukan/beraka/sumitsukisomewakebera.html
[1057/1323] 処理中: https://shiny-ace.com/zukan/fuedaika/yellowbandedsnapper.html
[1058/1323] 処理中: https://shiny-ace.com/zukan/hataka/somewakeminamihanadai.html
[1059/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/oosujiishimochi.html
[1060/1323] 処理中: https://shiny-ace.com/zukan/fusakasagoka/himesangokasago.html
[1061/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/

[1153/1323] 処理中: https://shiny-ace.com/zukan/chouchouuoka/sudarechouchouuo.html
[1154/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/cortezangelfish.html
[1155/1323] 処理中: https://shiny-ace.com/zukan/chouchouuoka/fuuraichouchouuo.html
[1156/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/talbotsdemoiselle.html
[1157/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/hirenagasuzumedai.html
[1158/1323] 処理中: https://shiny-ace.com/zukan/nizadaika/gomatenguhagimodoki.html
[1159/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/hiregurosuzumedai.html
[1160/1323] 処理中: https://shiny-ace.com/zukan/kaeruankouka/kaeruankoumodoki.html
[1161/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/aosujitenjikudai.html
[1162/1323] 処理中: https://shiny-ace.com/zukan/kaeruankouka/kaeruankou-hairy.html
[1163/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/himawarisuzumedai.html
[1164/1323] 処理中: https://shiny-ace.com/zukan/manjuudaika/mikadsukitubameuo.html
[1165/1323] 処理中: https://shiny-ace.com/z

[1254/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/allenscarodinalfish.html
[1255/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai3-.html
[1256/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai1-.html
[1257/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/shirikirurisuzumedai.html
[1258/1323] 処理中: https://shiny-ace.com/zukan/kuroyurihazeka/ogurokuroyurihaze.html
[1259/1323] 処理中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai2-.html
[1260/1323] 処理中: https://shiny-ace.com/zukan/kinchakudaika/tatejimakinchakudai.html
[1261/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/hawaianbicolorchromis.html
[1262/1323] 処理中: https://shiny-ace.com/zukan/suzumedaika/scissortaildamselfish.html
[1263/1323] 処理中: https://shiny-ace.com/zukan/kuroyurihazeka/shikonhatatatehaze.html
[1264/1323] 処理中: https://shiny-ace.com/zukan/itoyoridaika/threestripedwhiptail.html
[1265/1323] 処理中: https://shiny-ace.com/zukan/chouchouuoka/sumitsukitonosamadai.htm

In [6]:
import os
import re
import time
import csv
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# --- 設定 ---
BASE_URL = "https://shiny-ace.com/"
TXT_FILE = "リンク一覧.txt"
CSV_FILE = "fish_list.csv"  # 日本語書き換え＆並べ替え済みのCSV

# リンク一覧から「zukan/分類/ページ名.html」のパターンを抽出する正規表現
url_pattern = re.compile(r'(zukan\d*/[^/]+/[^/]+\.html)')

# --- 1. リンク一覧の読み込み ---
print("1. リンク一覧を読み込んでいます...")
extracted_paths = []
with open(TXT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        match = url_pattern.search(line)
        if match:
            extracted_paths.append(match.group(1))
print(f" -> 合計 {len(extracted_paths)} 件のページURLを検出しました。")

# --- 2. 既存のCSVを読み込み、順序維持用のリストと検索マップを作成 ---
print("\n2. 既存の並び替え済みCSVを読み込んで保護しています...")
fish_list_ordered = []  # CSVの並び順を完全に維持するためのリスト
fish_data_map = {}      # 名前をキーにして行データを更新するための辞書

with open(CSV_FILE, 'r', encoding='utf-8-sig') as f:
    reader = csv.reader(f)
    header = next(reader)  # ヘッダーを退避
    
    for row in reader:
        if not row:
            continue
        fish_name = row[0].strip().replace("　", "") # 紐付けのためにスペースを除去
        
        # 最初の5列（名前、英名、画像1、分類、解説）をベースとし、後ろに初期値としての欠損を追加
        base_data = row[:5]
        if len(base_data) < 5:
            base_data += [""] * (5 - len(base_data))
        
        # [名前, 英名, 画像1, 分類, 解説, image2の初期値, image3の初期値]
        fish_data_map[fish_name] = base_data + ["欠損", "欠損"]
        fish_list_ordered.append(fish_name)

print(f" -> CSVから {len(fish_list_ordered)} 件の魚データを登録しました。")

# --- 3. ページにアクセスし、名前ベースで画像名をマッピング（ダウンロードは無し） ---
print("\n3. 各個別ページへアクセスし、画像名の紐付けを開始します...")
for i, path in enumerate(extracted_paths, 1):
    page_url = BASE_URL + path
    print(f"[{i}/{len(extracted_paths)}] 解析中: {page_url}")

    try:
        res = requests.get(page_url, timeout=10)
        res.raise_for_status()
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')

        # ① 1枚目の時と同じロジックで「魚の名前」を正確に取得
        fish_name = ""
        h2_tag = soup.find('h2')
        if h2_tag:
            h2_text = h2_tag.get_text()
            if "英名：" in h2_text:
                fish_name = h2_text.split("英名：")[0].strip().replace("　", "")
            else:
                fish_name = h2_text.strip().replace("　", "")
        
        if not fish_name:
            h1_tag = soup.find('h1')
            if h1_tag:
                fish_name = h1_tag.contents[0].strip() if h1_tag.contents else h1_tag.get_text().strip()
                fish_name = fish_name.replace("　", "")

        # ② 複数figureに対応したロジックで2・3枚目の「画像ファイル名」を取得
        figure_tags = soup.find_all('figure', class_='gallery')
        gallery_imgs = []
        for fig in figure_tags:
            img_tag = fig.find('img')
            if img_tag and img_tag.get('src'):
                # 絶対URLに変換してから、末尾のファイル名（例: akaei2.jpg）を抽出
                img_filename = urljoin(page_url, img_tag.get('src')).split('/')[-1]
                gallery_imgs.append(img_filename)

        image2_name = gallery_imgs[1] if len(gallery_imgs) > 1 else "欠損"
        image3_name = gallery_imgs[2] if len(gallery_imgs) > 2 else "欠損"

        # ③ CSVから読み込んだマップと【魚の名前】で紐付けしてデータを更新
        if fish_name in fish_data_map:
            fish_data_map[fish_name][5] = image2_name  # 6列目(image2)を上書き
            fish_data_map[fish_name][6] = image3_name  # 7列目(image3)を上書き
        else:
            print(f"  --> [警告] ページ上の名前「{fish_name}」が既存のCSV内に見つかりません。")

    except Exception as e:
        print(f"  --> [エラー発生] {page_url} の解析中に問題が発生しました: {e}")
    
    # インターバル 0.1秒
    time.sleep(0.1)

# --- 4. 元の並び順のままCSVに上書き保存 ---
print("\n4. データをCSVファイルに再書き込みしています...")
new_header = ["魚の名前", "英名", "保存した画像ファイル名", "分類", "解説", "image2", "image3"]

with open(CSV_FILE, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(new_header)
    for name in fish_list_ordered:
        writer.writerow(fish_data_map[name])

print("\n[完了] 名前基準での正確な画像名マッピングが完了しました！CSVが更新されました。")

1. リンク一覧を読み込んでいます...
 -> 合計 1323 件のページURLを検出しました。

2. 既存の並び替え済みCSVを読み込んで保護しています...
 -> CSVから 1323 件の魚データを登録しました。

3. 各個別ページへアクセスし、画像名の紐付けを開始します...
[1/1323] 解析中: https://shiny-ace.com/zukan/ei/akaei.html
[2/1323] 解析中: https://shiny-ace.com/zukan/ei/yakkoei.html
[3/1323] 解析中: https://shiny-ace.com/zukan/beraka/ira.html
[4/1323] 解析中: https://shiny-ace.com/zukan/ajika/buri.html
[5/1323] 解析中: https://shiny-ace.com/zukan/hataka/kue.html
[6/1323] 解析中: https://shiny-ace.com/zukan/boraka/bora.html
[7/1323] 解析中: https://shiny-ace.com/zukan/ajika/maaji.html
[8/1323] 解析中: https://shiny-ace.com/zukan/taika/madai.html
[9/1323] 解析中: https://shiny-ace.com/zukan/ei/madaraei.html
[10/1323] 解析中: https://shiny-ace.com/zukan/ei/hirataei.html
[11/1323] 解析中: https://shiny-ace.com/zukan/ei/mobula1-.html
[12/1323] 解析中: https://shiny-ace.com/zukan/ajika/meaji.html
[13/1323] 解析中: https://shiny-ace.com/zukan/sugika/sugi.html
[14/1323] 解析中: https://shiny-ace.com/zukan/sabaka/cero.html
[15/1323] 解析中: https://shiny-

[127/1323] 解析中: https://shiny-ace.com/zukan/hazeka/otomehaze.html
[128/1323] 解析中: https://shiny-ace.com/zukan/beraka/otomebera.html
[129/1323] 解析中: https://shiny-ace.com/zukan/esoka/hoshinoeso.html
[130/1323] 解析中: https://shiny-ace.com/zukan/hazeka/oomonhaze.html
[131/1323] 解析中: https://shiny-ace.com/zukan/beraka/ogurobera.html
[132/1323] 解析中: https://shiny-ace.com/zukan/hazeka/kobanhaze.html
[133/1323] 解析中: https://shiny-ace.com/zukan/gonbeka/okigonbe.html
[134/1323] 解析中: https://shiny-ace.com/zukan/hataka/oomonhata.html
[135/1323] 解析中: https://shiny-ace.com/zukan/hazeka/colongoby.html
[136/1323] 解析中: https://shiny-ace.com/zukan/hazeka/oiranhaze.html
[137/1323] 解析中: https://shiny-ace.com/zukan/hataka/sakuradai.html
[138/1323] 解析中: https://shiny-ace.com/zukan/kisuka/shirogisu.html
[139/1323] 解析中: https://shiny-ace.com/zukan/beraka/yashabera.html
[140/1323] 解析中: https://shiny-ace.com/zukan/hazeka/odorihaze.html
[141/1323] 解析中: https://shiny-ace.com/zukan/hataka/houkihata.html
[142/1323]

[251/1323] 解析中: https://shiny-ace.com/zukan/gonbeka/kudagonbe.html
[252/1323] 解析中: https://shiny-ace.com/zukan/beraka/kujakubera.html
[253/1323] 解析中: https://shiny-ace.com/zukan/beraka/kusabibera.html
[254/1323] 解析中: https://shiny-ace.com/zukan/kochika/wanigochi.html
[255/1323] 解析中: https://shiny-ace.com/zukan/hazeka/noborihaze.html
[256/1323] 解析中: https://shiny-ace.com/zukan/himejika/umihigoi.html
[257/1323] 解析中: https://shiny-ace.com/zukan/ajika/hosohiraaji.html
[258/1323] 解析中: https://shiny-ace.com/zukan/hataka/yukatahata.html
[259/1323] 解析中: https://shiny-ace.com/zukan/hataka/madarahata.html
[260/1323] 解析中: https://shiny-ace.com/zukan/budaika/sujibudai.html
[261/1323] 解析中: https://shiny-ace.com/zukan/beraka/manabebera.html
[262/1323] 解析中: https://shiny-ace.com/zukan/hazeka/madarahaze.html
[263/1323] 解析中: https://shiny-ace.com/zukan/hazeka/nagasehaze.html
[264/1323] 解析中: https://shiny-ace.com/zukan/hataka/sarasahata.html
[265/1323] 解析中: https://shiny-ace.com/zukan/hataka/aonomehata.

[372/1323] 解析中: https://shiny-ace.com/zukan/budaika/nanyoubudai.html
[373/1323] 解析中: https://shiny-ace.com/zukan/utsuboka/kokeutsubo.html
[374/1323] 解析中: https://shiny-ace.com/zukan/hazeka/shimaorihaze.html
[375/1323] 解析中: https://shiny-ace.com/zukan/hataka/tsuchihozeri.html
[376/1323] 解析中: https://shiny-ace.com/zukan/hazeka/goldspotgoby.html
[377/1323] 解析中: https://shiny-ace.com/zukan/ittodaika/nijiebisu.html
[378/1323] 解析中: https://shiny-ace.com/zukan/hazeka/hagoromohaze.html
[379/1323] 解析中: https://shiny-ace.com/zukan/nizadaika/tenguhagi.html
[380/1323] 解析中: https://shiny-ace.com/zukan/hakofuguka/hakofugu.html
[381/1323] 解析中: https://shiny-ace.com/zukan/beraka/hagehirabera.html
[382/1323] 解析中: https://shiny-ace.com/zukan2/kamemoku/aoumigame.html
[383/1323] 解析中: https://shiny-ace.com/zukan/hazeka/garasuhaze5-.html
[384/1323] 解析中: https://shiny-ace.com/zukan/himejika/indohimeji.html
[385/1323] 解析中: https://shiny-ace.com/zukan/hazeka/yamabukihaze.html
[386/1323] 解析中: https://shiny-ace.

[491/1323] 解析中: https://shiny-ace.com/zukan/hataka/niramihanadai.html
[492/1323] 解析中: https://shiny-ace.com/zukan/budaika/kanmuribudai.html
[493/1323] 解析中: https://shiny-ace.com/zukan/umitanagoka/aotanago.html
[494/1323] 解析中: https://shiny-ace.com/zukan/hataka/panamagraysby.html
[495/1323] 解析中: https://shiny-ace.com/zukan/same/oguromejirozame.html
[496/1323] 解析中: https://shiny-ace.com/zukan/hataka/osyarehanadai.html
[497/1323] 解析中: https://shiny-ace.com/zukan/hazeka/blackeyedgoby.html
[498/1323] 解析中: https://shiny-ace.com/zukan/beraka/kazarikyuusen.html
[499/1323] 解析中: https://shiny-ace.com/zukan/hataka/harlequinbass.html
[500/1323] 解析中: https://shiny-ace.com/zukan/beraka/kisujikyuusen.html
[501/1323] 解析中: https://shiny-ace.com/zukan/hazeka/obakeinkohaze.html
[502/1323] 解析中: https://shiny-ace.com/zukan/isakika/blackmargate.html
[503/1323] 解析中: https://shiny-ace.com/zukan/beraka/hokurokyuusen.html
[504/1323] 解析中: https://shiny-ace.com/zukan/hazeka/kuroitohaze1-.html
[505/1323] 解析中: http

...

[832/1323] 解析中: https://shiny-ace.com/zukan/takasagoka/bananafusilier.html
[833/1323] 解析中: https://shiny-ace.com/zukan/fuguka/kazarikinchakufugu.html
[834/1323] 解析中: https://shiny-ace.com/zukan/harisenbonka/ishigakifugu.html
[835/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/rurisuzumedai.html
[836/1323] 解析中: https://shiny-ace.com/zukan/isoginpoka/oogonnijiginpo.html
[837/1323] 解析中: https://shiny-ace.com/zukan/beraka/munatenberadamashi.html
[838/1323] 解析中: https://shiny-ace.com/zukan/toragisuka/madaratoragisu.html
[839/1323] 解析中: https://shiny-ace.com/zukan/hazeka/candycanedwarfgoby.html
[840/1323] 解析中: https://shiny-ace.com/zukan/fusakasagoka/ukkarikasago.html
[841/1323] 解析中: https://shiny-ace.com/zukan/matsukasauoka/matsukasauo.html
[842/1323] 解析中: https://shiny-ace.com/zukan/isakika/hiregurokosyoudai.html
[843/1323] 解析中: https://shiny-ace.com/zukan/ittodaika/hireguroittodai.html
[844/1323] 解析中: https://shiny-ace.com/zukan/kinchakudaika/kinchakudai.html
[845/1323] 解析中: https://s

[941/1323] 解析中: https://shiny-ace.com/zukan/kinchakudaika/akaharayakko.html
[942/1323] 解析中: https://shiny-ace.com/zukan/mongarakawahagika/kumadori.html
[943/1323] 解析中: https://shiny-ace.com/zukan/ittodaika/kuroobomatsukasa.html
[944/1323] 解析中: https://shiny-ace.com/zukan/isoginpoka/segmentedblenny.html
[945/1323] 解析中: https://shiny-ace.com/zukan/kinchakudaika/inadumayakko.html
[946/1323] 解析中: https://shiny-ace.com/zukan/isoginpoka/ishigakikaeruuo.html
[947/1323] 解析中: https://shiny-ace.com/zukan/isoginpoka/tategamikaeruuo.html
[948/1323] 解析中: https://shiny-ace.com/zukan/utsuboka/panamicgreenmoray.html
[949/1323] 解析中: https://shiny-ace.com/zukan/kintokidaika/minamikintoki.html
[950/1323] 解析中: https://shiny-ace.com/zukan/kaeruankouka/irokaeruankou.html
[951/1323] 解析中: https://shiny-ace.com/zukan/nezuppoka/minamikobunumeri.html
[952/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/imitatordamsel.html
[953/1323] 解析中: https://shiny-ace.com/zukan/asahiginpoka/giantkelpfish.html
[954/1323] 解

[1048/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/goldbellydamsel.html
[1049/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/coraldemoiselle.html
[1050/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/koganesuzumedai.html
[1051/1323] 解析中: https://shiny-ace.com/zukan/fusakasagoka/hanaminokasago.html
[1052/1323] 解析中: https://shiny-ace.com/zukan/agoamadaika/variablejawfish.html
[1053/1323] 解析中: https://shiny-ace.com/zukan/kokeginpoka/hadakakokeginpo.html
[1054/1323] 解析中: https://shiny-ace.com/zukan/kinchakudaika/zebraangelfish.html
[1055/1323] 解析中: https://shiny-ace.com/zukan/mongarakawahagika/akamongara.html
[1056/1323] 解析中: https://shiny-ace.com/zukan/beraka/sumitsukisomewakebera.html
[1057/1323] 解析中: https://shiny-ace.com/zukan/fuedaika/yellowbandedsnapper.html
[1058/1323] 解析中: https://shiny-ace.com/zukan/hataka/somewakeminamihanadai.html
[1059/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/oosujiishimochi.html
[1060/1323] 解析中: https://shiny-ace.com/zukan/fusakasagoka/

[1152/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/sangiruishimochi.html
[1153/1323] 解析中: https://shiny-ace.com/zukan/chouchouuoka/sudarechouchouuo.html
[1154/1323] 解析中: https://shiny-ace.com/zukan/kinchakudaika/cortezangelfish.html
[1155/1323] 解析中: https://shiny-ace.com/zukan/chouchouuoka/fuuraichouchouuo.html
[1156/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/talbotsdemoiselle.html
[1157/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/hirenagasuzumedai.html
[1158/1323] 解析中: https://shiny-ace.com/zukan/nizadaika/gomatenguhagimodoki.html
[1159/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/hiregurosuzumedai.html
[1160/1323] 解析中: https://shiny-ace.com/zukan/kaeruankouka/kaeruankoumodoki.html
[1161/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/aosujitenjikudai.html
[1162/1323] 解析中: https://shiny-ace.com/zukan/kaeruankouka/kaeruankou-hairy.html
[1163/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/himawarisuzumedai.html
[1164/1323] 解析中: https://shiny-ace.com/z

[1253/1323] 解析中: https://shiny-ace.com/zukan/fusakasagoka/shimahimeyamanokami.html
[1254/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/allenscarodinalfish.html
[1255/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai3-.html
[1256/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai1-.html
[1257/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/shirikirurisuzumedai.html
[1258/1323] 解析中: https://shiny-ace.com/zukan/kuroyurihazeka/ogurokuroyurihaze.html
[1259/1323] 解析中: https://shiny-ace.com/zukan/tenjikudaika/sukashitenjikudai2-.html
[1260/1323] 解析中: https://shiny-ace.com/zukan/kinchakudaika/tatejimakinchakudai.html
[1261/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/hawaianbicolorchromis.html
[1262/1323] 解析中: https://shiny-ace.com/zukan/suzumedaika/scissortaildamselfish.html
[1263/1323] 解析中: https://shiny-ace.com/zukan/kuroyurihazeka/shikonhatatatehaze.html
[1264/1323] 解析中: https://shiny-ace.com/zukan/itoyoridaika/threestripedwhiptail.html